# 월요일 효과 (Monday Effect) 연구

**가설**: 주식 시장에서 월요일은 다른 요일보다 수익률이 낮거나 (또는 높거나) 비정상적인 패턴이 있다.

이 노트북에서 할 것:
1. 요일별 평균 수익률 분포 시각화
2. 월요일 매수 → 금요일 매도 전략 백테스트 (`from_signals`)
3. 롤링 백테스트로 안정성 확인
4. 통계적 유의성 검증 (t-test + bootstrap)
5. 레짐별 성과 분석 (강세장 vs 약세장)
6. 파라미터 스윕: 매수 요일 × 매도 요일 모든 조합
7. 종합 tearsheet


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from ginlix_backtest.engine import portfolio, signals
from ginlix_backtest.engine import rolling as rolling_bt
from ginlix_backtest.analysis import regime as regime_mod
from ginlix_backtest.analysis import stats as stat_mod
from ginlix_backtest.analysis import tearsheet
from ginlix_backtest.engine import sweep
from ginlix_backtest import indicators

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. 데이터 로드

SPY (S&P500 ETF) 를 사용한다. yfinance 에서 직접 받아도 되고,  
이미 스냅샷을 만들어뒀다면 DataFeed 를 써도 된다.

여기서는 yfinance 직접 사용 (스냅샷 없이도 실행 가능).

In [ ]:
SYMBOL = 'SPY'
START  = '2005-01-01'
END    = '2024-12-31'

raw = yf.download(SYMBOL, start=START, end=END, auto_adjust=True, progress=False)
close = raw['Close'].squeeze()  # pd.Series
close.index = pd.to_datetime(close.index).tz_localize('UTC')

# SPY is also the benchmark
benchmark = close.copy()

print(f'{SYMBOL}: {close.index[0].date()} ~ {close.index[-1].date()}, {len(close):,} bars')
close.tail(3)

## 2. 요일별 평균 수익률 분석

먼저 전략을 짜기 전에, 데이터 자체를 탐색한다.

In [ ]:
daily_ret = close.pct_change().dropna()

dow_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}
dow_returns = daily_ret.groupby(daily_ret.index.dayofweek)

stats_df = pd.DataFrame({
    'mean': dow_returns.mean() * 100,
    'std':  dow_returns.std()  * 100,
    'median': dow_returns.median() * 100,
    'positive_pct': dow_returns.apply(lambda x: (x > 0).mean() * 100),
    'count': dow_returns.count(),
}).rename(index=dow_map)

display(stats_df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in stats_df['mean']]
stats_df['mean'].plot.bar(ax=axes[0], color=colors, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Average Daily Return by Day of Week (%)')
axes[0].set_ylabel('Mean Return (%)')
axes[0].tick_params(axis='x', rotation=0)

stats_df['positive_pct'].plot.bar(ax=axes[1], color='#3498db', edgecolor='white')
axes[1].axhline(50, color='red', linestyle='--', linewidth=0.8, label='50%')
axes[1].set_title('Win Rate by Day of Week (%)')
axes[1].set_ylabel('% Days Positive')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.suptitle(f'{SYMBOL} Day-of-Week Effect ({START} ~ {END})', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. 전략 백테스트: 월요일 매수 → 금요일 매도

`signals.weekday_pattern()` 을 사용한다.  
시그널은 자동으로 next-bar shift 적용 → **look-ahead 없음**.

In [ ]:
entries, exits = signals.weekday_pattern(close, buy_day='monday', sell_day='friday')

result = portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    benchmark_close=benchmark,
    symbol=SYMBOL,
)

print(result)
print()
result.plot()

In [ ]:
# 기본 지표 테이블
display(tearsheet.metrics_table(result))

## 4. 롤링 백테스트 — 전략 안정성 확인

단일 구간 성과는 운이 좋은 구간을 골랐을 수도 있다.  
5년 윈도우 × 3개월 스텝으로 여러 구간에서 반복 측정한다.

In [ ]:
def monday_strategy(price_slice: pd.Series):
    ent, ext = signals.weekday_pattern(price_slice, buy_day='monday', sell_day='friday')
    return portfolio.from_signals(price_slice, ent, ext)

rolling_result = rolling_bt.backtest(
    strategy_fn=monday_strategy,
    prices=close,
    window='5Y',
    step='3M',
    min_bars=60,
)

print(f'총 {len(rolling_result.windows)} 윈도우')
print(f'안정성 스코어 (양의 Sharpe 비율): {rolling_result.stability_score():.0%}')
display(rolling_result.summary().head(8))

In [ ]:
rolling_result.plot_metric('sharpe')

In [ ]:
print('최악의 5개 윈도우:')
display(rolling_result.worst_windows(n=5))

print('\n최고의 5개 윈도우:')
display(rolling_result.best_windows(n=5))

## 5. 통계적 유의성 검증

수익률이 0보다 유의미하게 크다고 할 수 있는가?

- **t-test**: 고전적 통계 검정
- **Bootstrap**: 가정 없는 재표본 방식
- **Monte Carlo**: 랜덤 진입 전략 대비 얼마나 좋은가?

In [ ]:
strat_returns = result.returns

# t-test
tt = stat_mod.ttest(strat_returns)
print('=== t-test ===')
print(tt)

# Bootstrap
bs = stat_mod.bootstrap_pvalue(strat_returns, n_simulations=5000)
print('\n=== Bootstrap ===')
print(bs)

In [ ]:
# Monte Carlo — 랜덤 진입 전략 null 분포
mc = stat_mod.monte_carlo(
    prices=close,
    n_simulations=1000,
    holding_days=5,   # 월~금 = 5거래일 보유와 같은 조건
    observed_sharpe=result.metrics['sharpe'],
)
print(mc)
print(f'전략 Sharpe 백분위: {mc.percentile():.1f}th percentile')
print(f'p-value: {mc.p_value():.4f}')
mc.plot()

## 6. 레짐별 성과 분석

월요일 효과는 강세장/약세장에서 다르게 작동할 수 있다.

In [ ]:
ra = regime_mod.RegimeAnalysis(returns=strat_returns, prices=close)

# MA200 기반 강세/약세장
trend_report = ra.by_trend(ma_window=200)
print(f'=== {trend_report.dimension} ===')
display(trend_report.summary())
trend_report.plot()

In [ ]:
# VIX 데이터 가져오기 (옵션)
try:
    vix_raw = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)
    vix = vix_raw['Close'].squeeze()
    vix.index = pd.to_datetime(vix.index).tz_localize('UTC')

    vix_report = ra.by_vix(vix)
    print(f'=== {vix_report.dimension} ===')
    display(vix_report.summary())
    vix_report.plot()
except Exception as e:
    print(f'VIX 데이터 없음: {e}')

## 7. 파라미터 스윕: 매수 요일 × 매도 요일

월요일 특별히 좋은지, 아니면 다른 요일 조합이 더 좋은지 비교.

> 경고: 이 결과로 "최적" 파라미터를 고르는 것은 과최적화.  
> 결과의 분포(안정성 지도)를 보는 것이 목적이다.

In [ ]:
DAYS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']

def dow_strategy(prices, buy_day, sell_day):
    if buy_day == sell_day:
        # 같은 요일은 의미 없음
        raise ValueError('buy_day == sell_day')
    ent, ext = signals.weekday_pattern(prices, buy_day=buy_day, sell_day=sell_day)
    return portfolio.from_signals(prices, ent, ext)

sweep_result = sweep.grid_search(
    strategy_fn=dow_strategy,
    prices=close,
    param_grid={'buy_day': DAYS, 'sell_day': DAYS},
    metric='sharpe',
    extra_metrics=['cagr', 'max_dd'],
)

print(f'안정성 지도 (양의 Sharpe 비율): {sweep_result.stability_map("buy_day", "sell_day"):.0%}')
print()
display(sweep_result.best(n=10))

In [ ]:
sweep_result.heatmap('buy_day', 'sell_day', metric='sharpe')

## 8. 종합 Tearsheet

최종 결과 종합 리포트.

In [ ]:
tearsheet.plot(
    result,
    benchmark_close=benchmark,
    title=f'월요일 효과 전략 — {SYMBOL} ({START}~{END})',
)

## 결론

| 항목 | 결과 |
|---|---|
| 단순 수익성 | 위 metrics_table 참조 |
| 안정성 (롤링) | `rolling_result.stability_score()` |
| 통계적 유의성 | t-test p-value, bootstrap 95% CI |
| 레짐 민감성 | 강세/약세장, 고VIX/저VIX 성과 차이 |
| 과최적화 위험 | 파라미터 스윕 분포로 확인 |

**해석 팁**
- 단순히 Sharpe 높다고 좋은 전략이 아니다.
- 안정성 스코어 70% 이상, p-value < 0.05, 파라미터 민감도 낮음 → 세 가지 동시에 충족해야 신뢰할 수 있다.
- 특정 레짐에서만 작동한다면, 레짐 필터를 추가하는 것이 합리적이다.
